<a href="https://colab.research.google.com/github/IgorKovacevicENNOH/ENNOH_Modelling/blob/main/Reading_OFF_PEMMDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction
packages installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)


Mounted at /content/drive


In [ ]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import json

ROOT_DIR = os.getcwd()
PROJECT_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model")


In [ ]:
sys.path.append(PROJECT_DIR)

from modules.getting_input_data import get_input_data

input_file_name = "input_file.xlsx"
input_data = get_input_data(PROJECT_DIR, input_file_name)

File found at: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/input_file.xlsx
The input data has been imported.


In [ ]:
project_name = input_data["project_name"]
zones = json.loads(input_data["zones"])
scenario =  input_data["scenario"]
year = input_data["year"]
DATA_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]), input_data["raw_data_dir"])
output_dir = os.path.join(PROJECT_DIR,str(input_data["data_set"]),input_data['inter_dir'],input_data["project_name"],input_data["scenario"], str(input_data["year"]))

capacities = {}

# Reading Excel files

## Electricity

In [ ]:
# Construct the file path for 'input_file.xlsx' directly within PROJECT_DIR
excel_file_in_project_dir = os.path.join(DATA_DIR,'Nodes','LIST OF NODES.xlsx')
# Corrected target sheet name based on user's latest input
target_sheet_name = 'Electricity_Offshore'

print(f"Attempting to open Excel file: {excel_file_in_project_dir}")

try:
    # Try to read the Excel file and get its sheet names
    xls = pd.ExcelFile(excel_file_in_project_dir)
    print(f"Successfully opened {excel_file_in_project_dir}.")
    sheet_names = xls.sheet_names
    print(f"Available sheets in '{os.path.basename(excel_file_in_project_dir)}': {sheet_names}")

    if target_sheet_name in sheet_names:
        print(f"Sheet '{target_sheet_name}' found. Loading it into a DataFrame...")
        nodes_df = pd.read_excel(xls, sheet_name=target_sheet_name)
        print(f"Successfully loaded '{target_sheet_name}' from {excel_file_in_project_dir}:")
        display(nodes_df.head())
    else:
        print(f"Sheet '{target_sheet_name}' was not found in '{os.path.basename(excel_file_in_project_dir)}'.")
        print("Please specify the correct sheet name or clarify if 'List of Nodes.xlsx' is a different file.")
        if sheet_names:
            first_sheet = sheet_names[0]
            print(f"Loading the first sheet '{first_sheet}' instead for inspection.")
            nodes_df = pd.read_excel(xls, sheet_name=first_sheet)
            display(nodes_df.head())
        else:
            print(f"No sheets found in {excel_file_in_project_dir}.")

except FileNotFoundError:
    print(f"Error: The file {excel_file_in_project_dir} was not found.")
    print("Please ensure the file exists at this path in your Google Drive.")
except Exception as e:
    print(f"An error occurred while reading the Excel file: {e}")

Attempting to open Excel file: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/input_data/Nodes/LIST OF NODES.xlsx
Successfully opened /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/input_data/Nodes/LIST OF NODES.xlsx.
Available sheets in 'LIST OF NODES.xlsx': ['Electricity', 'Electricity_Offshore', 'H2_Demand', 'H2_Bottlenecks', 'H2_Imports', 'H2_Offshore']
Sheet 'Electricity_Offshore' found. Loading it into a DataFrame...
Successfully loaded 'Electricity_Offshore' from /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/input_data/Nodes/LIST OF NODES.xlsx:


,NODE
0,BEO1_OFF
1,BEO2_OFF
2,DEKF_OFF
3,DKB2_OFF
4,DKBH_OFF


In [ ]:
print(f"Number of nodes: {len(nodes_df)}")

Number of nodes: 54


In [ ]:
offshore_name_df = nodes_df.rename(columns={'NODE': 'Offshore_names'})
wind_data_collection = {}

for offshore_name in offshore_name_df['Offshore_names']:
    file_path = os.path.join(DATA_DIR, f"PEMMDB_2.X/{year}/PEMMDB_{offshore_name}_NationalTrends_{year}.xlsx")

    try:
        # Read the 'Wind' sheet from the Excel file
        wind_df = pd.read_excel(file_path, sheet_name='Wind')
        wind_data_collection[offshore_name] = wind_df
        print(f"Successfully loaded 'Wind' data for {offshore_name}.")
    except FileNotFoundError:
        print(f"Error: File not found for {offshore_name} at {file_path}")
    except Exception as e:
        print(f"An error occurred while reading 'Wind' sheet for {offshore_name}: {e}")

print("\n--- Collected Wind Data Preview ---")
if wind_data_collection:
    # Display the head of the first collected DataFrame as an example
    first_offshore_name = list(wind_data_collection.keys())[0]
else:
    print("No Wind data was collected.")

Successfully loaded 'Wind' data for BEO1_OFF.
Successfully loaded 'Wind' data for BEO2_OFF.
Successfully loaded 'Wind' data for DEKF_OFF.
Successfully loaded 'Wind' data for DKB2_OFF.
Error: File not found for DKBH_OFF at /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/input_data/PEMMDB_2.X/2030/PEMMDB_DKBH_OFF_NationalTrends_2030.xlsx
Error: File not found for DKBF_OFF at /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/input_data/PEMMDB_2.X/2030/PEMMDB_DKBF_OFF_NationalTrends_2030.xlsx
Successfully loaded 'Wind' data for DKHE_OFF.
Successfully loaded 'Wind' data for DKK2_OFF.
Successfully loaded 'Wind' data for DKKA_OFF.
Successfully loaded 'Wind' data for DKKF_OFF.
Successfully loaded 'Wind' data for DKN1_OFF.
Successfully loaded 'Wind' data for DKN2_OFF.
Successfully loaded 'Wind' data for DKN3_OFF.
Successfully loaded 'Wind' data for DKN4_OFF.
Successfully loaded 'Wind' data for DKN5_OFF.
Successfully loaded 'Wind' data for 

In [ ]:
num_wind_profiles = len(wind_data_collection)
print(f"Number of wind profiles found: {num_wind_profiles}")

Number of wind profiles found: 46


In [ ]:
extracted_wind_metadata = {}

for offshore_name, wind_df in wind_data_collection.items():
    # Extract rows A8 to B21 (0-indexed: rows 7 to 20, columns 0 and 1)
    # .iloc is used for integer-location based indexing
    metadata_slice = wind_df.iloc[7:21, [0, 1]].copy()

    # Set column A as the index for key-value pairs
    metadata_slice.columns = ['Key', 'Value']
    metadata_slice.set_index('Key', inplace=True)

    # Convert to dictionary and store
    extracted_wind_metadata[offshore_name] = metadata_slice['Value'].to_dict()

print("\n--- Extracted Wind Metadata Preview ---")
if extracted_wind_metadata:
    # Display metadata for the first offshore name as an example
    first_offshore_name_with_data = list(extracted_wind_metadata.keys())[0]
    print(f"Metadata for '{first_offshore_name_with_data}':")
    for key, value in extracted_wind_metadata[first_offshore_name_with_data].items():
        print(f"  {key}: {value}")
else:
    print("No metadata was extracted.")


--- Extracted Wind Metadata Preview ---
Metadata for 'BEO1_OFF':
  Installed capacities Onshore wind e-market (GW):: nan
  Installed capacities Onshore wind dedicated (GW):: nan
  Installed capacities Onshore wind Shared RES - H2Z1 (GW):: nan
  Installed capacities Onshore wind Shared RES - H2Z2 (GW):: nan
  Installed capacities Offshore wind Total (GW):: nan
  Installed capacities Offshore wind e-market - hub ready (GW):: nan
  Installed capacities Offshore wind dedicated - hub ready (GW):: nan
  Installed capacities Offshore wind Shared RES - H2Z1 - hub ready (GW):: nan
  Installed capacities Offshore wind Shared RES - H2Z2 - hub ready (GW):: nan
  Installed capacities Offshore wind e-market - radial (GW):: nan
  Installed capacities Offshore wind dedicated - radial (GW):: nan
  Installed capacities Offshore wind Shared RES - H2Z1 - radial (GW):: nan
  Installed capacities Offshore wind Shared RES - H2Z2 - radial (GW):: nan


## H2

In [ ]:
# Construct the file path for 'input_file.xlsx' directly within PROJECT_DIR
excel_file_in_project_dir = os.path.join(DATA_DIR,'Nodes','LIST OF NODES.xlsx')
# Corrected target sheet name based on user's latest input
target_sheet_name = 'H2_Offshore'


In [ ]:
# Missing DK offshore wind farms on H2 offshore list
add_DK_offshore = ['DKKF_OFF','DKKA_OFF','DKN1_OFF','DKN2_OFF','DKN3_OFF','DKN4_OFF','DKN5_OFF','DKN6_OFF','DKN7_OFF','DKN8_OFF','DKN9_OFF','DKNS_OFF']

In [ ]:
try:
    # Try to read the Excel file and get its sheet names
    xls = pd.ExcelFile(excel_file_in_project_dir)
    print(f"Successfully opened {excel_file_in_project_dir}.")
    sheet_names = xls.sheet_names
    print(f"Available sheets in '{os.path.basename(excel_file_in_project_dir)}': {sheet_names}")

    if target_sheet_name in sheet_names:
        print(f"Sheet '{target_sheet_name}' found. Loading it into a DataFrame...")
        h2_offshore_df = pd.read_excel(xls, sheet_name=target_sheet_name)
        print(f"Successfully loaded '{target_sheet_name}' from {excel_file_in_project_dir}:")

        # Create a list from the first column of the DataFrame
        if not h2_offshore_df.empty and len(h2_offshore_df.columns) > 0:
            h2_offshore_list = h2_offshore_df.iloc[:, 0].tolist()
            print(f"\nSuccessfully created a list from the first column of '{target_sheet_name}'.")
        else:
            print(f"The sheet '{target_sheet_name}' is empty or has no columns.")
            h2_offshore_list = []

    else:
        print(f"Sheet '{target_sheet_name}' was not found in '{os.path.basename(excel_file_in_project_dir)}'.")
        print("Please specify the correct sheet name.")
        h2_offshore_list = []

except FileNotFoundError:
    print(f"Error: The file {excel_file_in_project_dir} was not found.")
    print("Please ensure the file exists at this path in your Google Drive.")
    h2_offshore_list = []
except Exception as e:
    print(f"An error occurred while reading the Excel file or creating the list: {e}")
    h2_offshore_list = []


Successfully opened /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/input_data/Nodes/LIST OF NODES.xlsx.
Available sheets in 'LIST OF NODES.xlsx': ['Electricity', 'Electricity_Offshore', 'H2_Demand', 'H2_Bottlenecks', 'H2_Imports', 'H2_Offshore']
Sheet 'H2_Offshore' found. Loading it into a DataFrame...
Successfully loaded 'H2_Offshore' from /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/input_data/Nodes/LIST OF NODES.xlsx:

Successfully created a list from the first column of 'H2_Offshore'.


In [ ]:
for node in add_DK_offshore:
    if node not in h2_offshore_list:
        h2_offshore_list.append(node)

In [ ]:
# Define targets and parameters
target_type = "Electrolyser e-market Z2"
param_names = [
    "Net maximum capacity (MW)", "Number of units", "Average efficiency",
    "H2 storage (GWh)", "Ramp up rate (MW/h)", "Ramp down rate (MW/h)",
    "Fixed generation reduction (% of max power output)"
]

consolidated_z2_data = {}

# Loop through nodes, read Excel, and filter data
print("Reading and filtering H2 electrolyser data...")
for h2_node in h2_offshore_list:
    base_node_name = h2_node.replace('h2', '')
    file_path = os.path.join(DATA_DIR, f"PEMMDB_2.X/{year}/PEMMDB_{base_node_name}_NationalTrends_{year}.xlsx")

    # Default to 0s if file is missing or data is not found
    node_params = [0] * len(param_names)
    try:
        df = pd.read_excel(file_path, sheet_name='Electrolyser')
        key_col = df.columns[0]
        param_cols = df.columns[2:9] # Columns 2 to 8 match the 7 parameters

        mask = df[key_col].astype(str).str.strip() == target_type
        if mask.any():
            node_params = df.loc[mask, param_cols].values[0].tolist()
    except Exception:
        pass # If file/sheet missing or error occurs, it silently keeps the default zeros

    # Map parameter names to the extracted values to store as a dictionary
    consolidated_z2_data[h2_node] = dict(zip(param_names, node_params))

Reading and filtering H2 electrolyser data...


# Saving data to file

In [ ]:
# Create the combined dictionary
capacities_offshore = {
    "wind": extracted_wind_metadata,
    "electrolyser": consolidated_z2_data
}

# Save the combined dictionary to a JSON file
output_file_combined = os.path.join(output_dir, "capacities_offshore.json")
os.makedirs(output_dir, exist_ok=True)
with open(output_file_combined, 'w') as f:
    json.dump(capacities_offshore, f, indent=4)

# Display Summary and Preview
print(f"\u2705 Combined offshore capacities (wind & electrolyser) saved to: {output_file_combined}")

print("\n--- Dictionary Structure ---")
for key in capacities_offshore.keys():
    print(f"- {key}: {len(capacities_offshore[key])} nodes")


✅ Combined offshore capacities (wind & electrolyser) saved to: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/intermediate_data/Europe/DE/2030/capacities_offshore.json

--- Dictionary Structure ---
- wind: 46 nodes
- electrolyser: 38 nodes
